# Getting started with Immune: protect, configure and test a real app

This notebook takes you from `pip install` to an app that is protected, configured and tested, in the order you would
adopt Immune in your own project. Every section has the same shape:

- **Goal:** what the section is for.
- **Do it:** the smallest code that shows it.
- **What you'll see:** what to look for in the output.
- **In your app:** the code or file to copy.

**Live or offline.** With `OPENAI_API_KEY` and `TYPESAFE_API_KEY` in your environment (or in the repository's `.env`),
the notebook calls OpenAI (`gpt-5.4-mini` by default) and Jev for real. Without them it runs offline: a scripted model
and scripted Jev answers stand in, so every cell still runs.

**Cost.** The saved run made 38 OpenAI calls for about one cent, and about 2,000 Jev tokens per screened call. A budget guard stops
live calls at `DEMO_MAX_CALLS` (default 60) or when all demo runs together reach `DEMO_BUDGET_USD` (default $3).

**Attacks** are never written in this notebook. Where one is shown, it is replayed from Immune's scenario library,
which reconstructs publicly reported incidents, and the model's side is the recorded reply. The live model is only
asked ordinary questions.

| # | Section |
| --- | --- |
| 0 | [Before you start](#0.-Before-you-start) |
| 1 | [Protect your first call](#1.-Protect-your-first-call) |
| 2 | [Read a verdict](#2.-Read-a-verdict) |
| 3 | [What Immune saw](#3.-What-Immune-saw) |
| 4 | [Name the parts of your app](#4.-Name-the-parts-of-your-app) |
| 5 | [Write immune.yaml](#5.-Write-immune.yaml) |
| 6 | [Choose a posture per site](#6.-Choose-a-posture-per-site) |
| 7 | [Change behaviour live](#7.-Change-behaviour-live) |
| 8 | [Keep secrets and private data in](#8.-Keep-secrets-and-private-data-in) |
| 9 | [Don't let the reply copy your prompt](#9.-Don't-let-the-reply-copy-your-prompt) |
| 10 | [Mark what you paste in](#10.-Mark-what-you-paste-in) |
| 11 | [Links in replies](#11.-Links-in-replies) |
| 12 | [Stream safely](#12.-Stream-safely) |
| 13 | [Tools and agents](#13.-Tools-and-agents) |
| 14 | [Your own protections: vaccines](#14.-Your-own-protections-vaccines) |
| 15 | [When something is blocked](#15.-When-something-is-blocked) |
| 16 | [See it in your tools](#16.-See-it-in-your-tools) |
| 17 | [Test it](#17.-Test-it) |
| 18 | [Go to production](#18.-Go-to-production) |
| 19 | [Copy it into your app](#19.-Copy-it-into-your-app) |

## 0. Before you start

**Goal:** install Immune in this notebook's kernel and load your keys.

If the kernel doesn't have Immune yet, run the next cell once and restart the kernel.

In [1]:
import importlib.util

if importlib.util.find_spec("immune") is None:
    %pip install -e "../..[dev,notebook,langsmith]"
else:
    print("Immune is installed in this kernel")

Immune is installed in this kernel


Keys are read from the environment, or from the repository's `.env` file if there is one. Only whether each key is
set is printed, never the key.

In [2]:
import json
import logging
import os
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import openai

import immune
from immune.testing import FakeProvider, FakeReply, FakeToolCall, LangSmithRecorder, MockSensor, ScenarioLibrary


def find_repository():
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "examples" / "showcase").is_dir():
            return folder
    return None


def load_env(path):
    """Read KEY=value lines into os.environ, without printing them or replacing variables that are already set."""
    if not path.is_file():
        return
    for line in path.read_text(encoding="utf-8").splitlines():
        name, separator, value = line.strip().partition("=")
        if separator and name.isidentifier() and name not in os.environ:
            os.environ[name] = value.strip().strip("'\"")


REPOSITORY = find_repository()
OFFLINE = os.environ.get("IMMUNE_NOTEBOOK_OFFLINE") == "1"
if REPOSITORY is not None and not OFFLINE:
    load_env(REPOSITORY / ".env")
LIVE = not OFFLINE and bool(os.environ.get("OPENAI_API_KEY"))
JEV = LIVE and bool(os.environ.get("TYPESAFE_API_KEY"))
for name in ("OPENAI_API_KEY", "TYPESAFE_API_KEY", "LANGSMITH_API_KEY"):
    value = os.environ.get(name, "")
    print(f"{name:<18} {'set' if value else 'missing'}")
print("running", "live" if LIVE else "offline: scripted model and scripted Jev answers")

OPENAI_API_KEY     set
TYPESAFE_API_KEY   set
LANGSMITH_API_KEY  missing
running live


A few helpers keep the rest short. You don't need them in your app:

- `chat(client, text)` sends one message to the ordering assistant used throughout, with a small output budget.
- `start(...)` (re)starts Immune in this notebook with a scratch state directory.
- `show(verdict)` prints what Immune decided and why.
- `BUDGET` counts live OpenAI calls and dollars, and stops before the limit.

In [3]:
MODEL = os.environ.get("OPENAI_MODEL", "gpt-5.4-mini")
MENU_BOT = (
    "You are the ordering assistant for Bob's Burgers. Help customers with the menu, orders, refunds, delivery "
    "and loyalty points. "
    "Menu: Classic $9, Double Stack $12, Garden Stack (vegetarian) $11, Fries $3, Lemonade $2.50. "
    "Full menu: https://bobsburgers.example/menu"
)
WORK = Path(tempfile.mkdtemp(prefix="immune-getting-started-"))
HTTP = __import__("httpx2" if int(openai.__version__.split(".")[0]) >= 3 else "httpx")
SPEND_FILE = Path(os.environ.get("DEMO_SPEND_FILE", Path.home() / ".immune" / "demo-spend.json"))
if not JEV and REPOSITORY is not None:
    sys.path.insert(0, str(REPOSITORY / "examples" / "showcase"))
    from showcase.offline import ScriptedModel, scripted_sensor


class Budget:
    """Stops live calls at DEMO_MAX_CALLS, or when all demo runs together reach DEMO_BUDGET_USD."""

    def __init__(self):
        self.calls, self.usd, self.jev_tokens = 0, 0.0, 0
        self.max_calls = int(os.environ.get("DEMO_MAX_CALLS", "60"))
        self.cap = float(os.environ.get("DEMO_BUDGET_USD", "3.00"))
        try:
            self.previous = float(json.loads(SPEND_FILE.read_text())["usd"])
        except (OSError, ValueError, KeyError):
            self.previous = 0.0

    def check(self):
        if LIVE and (self.calls >= self.max_calls or self.previous + self.usd >= self.cap):
            raise RuntimeError(f"budget reached: {self}")

    def charge(self, response):
        verdict = immune.verdict(response)
        self.jev_tokens += verdict.sensor.input_tokens if verdict else 0
        if not LIVE or response.usage is None:
            return
        self.calls += 1
        self.usd += response.usage.prompt_tokens * 0.75e-6 + response.usage.completion_tokens * 4.5e-6
        SPEND_FILE.parent.mkdir(parents=True, exist_ok=True)
        SPEND_FILE.write_text(json.dumps({"usd": round(self.previous + self.usd, 6)}))

    def __str__(self):
        if not LIVE:
            return "offline: no OpenAI spend"
        total = self.previous + self.usd
        return f"{self.calls} OpenAI calls, ${self.usd:.4f} this run, ${total:.4f} of ${self.cap:.2f} across runs"


BUDGET = Budget()


def fake_client(script=None):
    provider = FakeProvider(script or ScriptedModel())
    return openai.OpenAI(
        api_key="sk-offline", max_retries=0, http_client=HTTP.Client(transport=provider.transport(HTTP))
    )


def new_client():
    return openai.OpenAI(max_retries=1) if LIVE else fake_client()


def chat(client, text, system=MENU_BOT, history=(), charge=True, **options):
    if charge:
        BUDGET.check()
    messages = [{"role": "system", "content": system}, *history, {"role": "user", "content": text}]
    response = client.chat.completions.create(
        model=MODEL, messages=messages, reasoning_effort="none", max_completion_tokens=300, **options
    )
    if charge:
        BUDGET.charge(response)
    return response


def reply_of(response):
    message = response.choices[0].message
    return message.content or ""


def cli(*arguments, cwd=None, env=None):
    """Run the immune command line and print its output in the notebook."""
    result = subprocess.run(
        [sys.executable, "-m", "immune.cli.main", *map(str, arguments)],
        cwd=cwd,
        env=env,
        capture_output=True,
        text=True,
        check=False,
    )
    print((result.stdout + result.stderr).rstrip())


def start(**options):
    immune.shutdown()
    options.setdefault("state_dir", WORK / "state")
    if not JEV:
        options.setdefault("sensor", scripted_sensor())
    return immune.init(**options)


def show(verdict):
    if verdict is None:
        print("no verdict: Immune did not screen this call")
        return
    print(f"action={verdict.action.value}  would_action={verdict.would_action.value}  site={verdict.site}")
    for hit in verdict.hits:
        state = "enforced" if hit.enforced else "observed"
        evidence = f"  · {'; '.join(hit.evidence[:2])}" if hit.evidence else ""
        floor = f" {hit.floor}" if hit.floor else ""
        print(f"  {hit.threat:<32} p={hit.probability:.2f}  {hit.action.value:<9} {state}{floor}{evidence}")
    sensor = verdict.sensor
    print(f"  why: {verdict.explanation}")
    print(f"  jev: {sensor.name}, {sensor.calls} requests, {sensor.latency_ms:.0f} ms, {sensor.input_tokens} tokens")


print(f"model {MODEL}; scratch folder {WORK.name}; {BUDGET}")

model gpt-5.4-mini; scratch folder immune-getting-started-nb_40gm6; 0 OpenAI calls, $0.0000 this run, $0.0787 of $3.00 across runs


In [4]:
cli("doctor", "--state-dir", WORK / "state", *(["--live"] if JEV else []))

check             ok   detail
────────────────  ───  ─────────────────────────────────────────────────
python            yes  3.14.4
spec              yes  2026.10.1 (49 threats)
TYPESAFE_API_KEY  yes  set
typesafe-sdk      yes  0.7.2
openai            yes  3.19.2
anthropic         yes  1.8.0
google-genai      yes  2.25.0
httpx2            yes  2.13.1
httpx             yes  0.28.1
litellm           yes  not installed
langchain-core    yes  not installed
interception      yes  httpx2 (every transport), httpx (every transport)
vaccines          yes  0 loaded
state             yes  /tmp/immune-getting-started-nb_40gm6/state
jev               yes  jev-1.13.0 in 699 ms


**What you'll see:** `immune doctor` checks Python, the spec, your key, the SDKs, interception and the state directory.
With a Jev key it also measures one live Jev call. The outputs saved in this notebook come from a live run with
`gpt-5.4-mini`; yours will differ a little, because the model and Jev answer afresh.

**Recap:** Immune installed; keys loaded; doctor passes

## 1. Protect your first call

**Goal:** protect every LLM call in the process with one line.

In [5]:
start()
client = new_client()
reply = chat(client, "What's on the menu?")
print(reply_of(reply))
show(immune.verdict(reply))

Here’s the menu at Bob’s Burgers:

- Classic — $9
- Double Stack — $12
- Garden Stack (vegetarian) — $11
- Fries — $3
- Lemonade — $2.50

If you want, I can also help you place an order.
action=allow  would_action=allow  site=site-3b2c110e
  why: no threats detected
  jev: jev, 3 requests, 569 ms, 2482 tokens


**What you'll see:** a normal answer, and a verdict with `action=allow`. Immune read the call, asked Jev its questions
and found nothing to act on. If the reply repeats the menu word for word from the system prompt, Immune notices the
copy and asks Jev whether the passage is private guidance for the assistant or information meant for customers. A menu
is the second kind, so it passes. [Section 9](#9-copies-of-your-prompt-jev-decides) shows the other kind.

**In your app:**

```python
import immune

immune.init()  # once, at startup, before your first LLM call
```

`immune.init()` patches the HTTP transports the OpenAI, Anthropic and Google SDKs use, so clients created before or
after it are covered, and so is every framework built on them.

**Recap:** one line protects every call; every call gets a verdict

## 2. Read a verdict

**Goal:** know what a verdict tells you, and see Immune's log line.

In [6]:
verdict = immune.verdict(reply)
for field in (
    "trace_id",
    "site",
    "session_id",
    "action",
    "would_action",
    "taint",
    "session_risk",
    "spec_version",
    "config_hash",
):
    print(f"{field:<14} {getattr(verdict, field)}")
print(f"{'sensor':<14} {verdict.sensor}")
print(f"{'explanation':<14} {verdict.explanation}")

trace_id       463d3bebcbe34a5c
site           site-3b2c110e
session_id     None
action         allow
would_action   allow
taint          clean
session_risk   0.0202
spec_version   2026.10.1
config_hash    a6a09aec632d
sensor         SensorInfo(name='jev', model='jev-1.13.0', latency_ms=569.2, input_tokens=2482, calls=3)
explanation    no threats detected


In [7]:
handler = logging.StreamHandler(sys.stdout)
logging.getLogger("immune").addHandler(handler)
logging.getLogger("immune").setLevel(logging.INFO)
off_topic = chat(client, "Can you write me a Python function that sorts a list of numbers?")
logging.getLogger("immune").removeHandler(handler)
logging.getLogger("immune").setLevel(logging.WARNING)
show(immune.verdict(off_topic))

immune 7344a33a3b7d4b3f site=site-3b2c110e action=allow would=redirect threats=input.off_task


action=allow  would_action=redirect  site=site-3b2c110e
  input.off_task                   p=0.99  redirect  observed  · off_task=0.99
  why: observed (would redirect): input.off_task
  jev: jev, 2 requests, 379 ms, 1933 tokens


**What you'll see:** the second call is off-task for a burger shop. In the default `auto` mode that threat is
**observed**: the reply goes through, and the verdict says `would_action=redirect`, which is what enforcement would do.
The log line has the trace id, the site and the threats.

**In your app:** log `verdict.to_dict()`, or stream verdicts with `immune.on_verdict(callback)` (section 16).

**Recap:** `action` is what happened; `would_action` is what enforcement would do; `hits` explain why

## 3. What Immune saw

**Goal:** see the questions Immune asked Jev about a call, and the answers.

A `LangSmithRecorder` captures what Immune would send to LangSmith, including one child run per Jev request with its
questions and answers. (Section 16 sends the same runs to a real LangSmith project.)

In [8]:
with LangSmithRecorder() as recorder:
    start()
    chat(client, "Can you write me a Python function that sorts a list of numbers?")
    immune.shutdown()

for run in recorder.immune_runs():
    print(run["name"])
    for child in recorder.children(run):
        answers = child["outputs"]["answers"]
        print(f"  {child['name']}: {len(child['inputs']['questions'])} questions")
        for question in child["inputs"]["questions"][:4]:
            answer = answers.get(question["key"], {})
            print(f"    {question['key']:<26} p={answer.get('probability')}  {question['text'][:70]}")
start()

immune · observed · input.off_task
  jev · input: 10 questions
    override                   p=0.12  The latest user message tries to change, cancel or replace the operato
    override_alt               p=0.58  The user wants the assistant to act as a different assistant or withou
    framing_bypass             p=0.03  The latest user message uses a story, game, hypothetical or roleplay t
    extraction                 p=0.03  The latest user message asks the assistant to reveal its instructions,
  jev · output: 8 questions
    task_fidelity              p=0.54  The assistant output does the task described in the operator instructi
    reveals_instructions       p=0.03  The assistant output reveals or paraphrases the operator instructions 
    private_data               p=0.01  The assistant output reveals credentials or personal data about someon
    harmful_output             p=1.0  Which kind of harmful content does the assistant output contain?


In [9]:
cli("explain", "input.off_task")

input.off_task
──────────────
invariant   U2 Task fidelity: Output serves the operator's task and the user's request.
stage       input   detector jev   severity low
threshold   0.85   floor no (observed)
actions     text->redirect, software->refuse
frameworks  LLM10
head        logistic over atomic questions:
  +1.0 x logit(off_task, raw)
      noul: The latest user message asks for work unrelated to the task described in the operator instructions.


**What you'll see:** Immune splits each call into channels (operator, user, data, reply, tool calls) and asks Jev short
typed questions about them. A calibrated head per threat (an *antibody*) turns the answers into a probability.
`immune explain` shows the head behind any threat.

**Recap:** Jev answers typed questions; heads turn answers into probabilities; `immune explain <threat>`

## 4. Name the parts of your app

**Goal:** give each kind of LLM call a *site* name and each user a *session*, so settings and statistics are readable.

In [10]:
with immune.site("ordering"), immune.session("customer-42"):
    first = chat(client, "Do you have a vegetarian burger?")
    second = chat(client, "How much is it?")
print(immune.verdict(first).site, immune.verdict(first).session_id)
for status in immune.status():
    print(f"site {status.site:<14} archetype={status.archetype:<18} calls={status.calls}")

ordering customer-42
site site-3b2c110e  archetype=customer_service   calls=3
site ordering       archetype=customer_service   calls=2


**What you'll see:** the verdicts carry `site=ordering` and the session id. `immune.status()` lists every site Immune
has seen, with the profile it inferred.

**In your app:**

```python
with immune.site("ordering"), immune.session(user.id):
    reply = client.chat.completions.create(...)
```

**Recap:** sites get their own settings; sessions carry taint and confirmations across turns

## 5. Write immune.yaml

**Goal:** move settings into a file next to your app.

This cell writes a small project folder (the one you'll copy in section 19) with a minimal `immune.yaml`, validates
it, and restarts Immune with it.

In [11]:
PROJECT = WORK / "my-app"
(PROJECT / "vaccines").mkdir(parents=True)
(PROJECT / "immune.yaml").write_text(
    """\
# Immune settings for my app. Relative paths are relative to this file.
mode: auto

vaccines:
  paths: [vaccines/]

sites:
  ordering:
    user_facing: true
    enforce: [input.off_task]
  ops:
    user_facing: false
""",
    encoding="utf-8",
)
print((PROJECT / "immune.yaml").read_text())
cli("config", "validate", PROJECT / "immune.yaml")
start(config=PROJECT / "immune.yaml")
print("sites configured:", sorted(immune.runtime().settings.sites))

# Immune settings for my app. Relative paths are relative to this file.
mode: auto

vaccines:
  paths: [vaccines/]

sites:
  ordering:
    user_facing: true
    enforce: [input.off_task]
  ops:
    user_facing: false



valid configuration (mode=auto, 2 sites)
sites configured: ['ops', 'ordering']


**What you'll see:** the file validates, and Immune restarts with two configured sites.

**In your app:** put `immune.yaml` next to your app, in the directory it starts from, and call `immune.init()` with no
arguments; it finds the file. Or point `IMMUNE_CONFIG` at it. Add editor completion with
`immune config schema > immune.schema.json` and a `# yaml-language-server: $schema=./immune.schema.json` first line.
The [setup guide](../../docs/guides/setup.md) shows a complete project layout.

**Recap:** `immune.yaml` next to the app; `immune config validate`; relative paths are relative to the file

## 6. Choose a posture per site

**Goal:** see how the mode and a site's settings decide what is enforced.

The same off-task request goes to two sites: `ordering` enforces `input.off_task` in `immune.yaml`, and `ops` doesn't.
Then the global mode changes.

In [12]:
OFF_TASK = "Can you write me a Python function that sorts a list of numbers?"
rows = []
for mode in ("auto", "strict", "observe"):
    immune.configure(mode=mode)
    for site in ("ordering", "ops"):
        with immune.site(site):
            verdict = immune.verdict(chat(client, OFF_TASK))
        rows.append((mode, site, verdict.action.value, verdict.would_action.value))
immune.configure(mode="auto")
print(f"{'mode':<9}{'site':<10}{'action':<10}would_action")
for row in rows:
    print(f"{row[0]:<9}{row[1]:<10}{row[2]:<10}{row[3]}")

mode     site      action    would_action
auto     ordering  redirect  redirect
auto     ops       allow     redirect
strict   ordering  redirect  redirect
strict   ops       redirect  redirect
observe  ordering  redirect  redirect
observe  ops       allow     redirect


**What you'll see:**

| Mode | `ordering` (enforces off-task) | `ops` |
| --- | --- | --- |
| `auto` | redirect | allow; `would_action` shows what enforcement would do |
| `strict` | redirect | redirect |
| `observe` | redirect: a site's `enforce` list wins over the mode | allow; `would_action` as in `auto` |

(The live model and Jev decide what is off-task, so a row can differ.) In `auto`, the floor (F1–F10) is always
enforced, and other detectors are promoted per site once their false alarms are provably rare.

Live, you will also see **observed** hits on ordinary replies, such as `output.follows_data_instructions` on an
answer that quotes a help article. That is why detectors start observed: their default weights are provisional.
Calibrate them on your own labeled traffic (`immune evidence fit`, see the
[developer guide](../../docs/guides/developer-guide.md#78-train-the-antibodies-heads)) before enforcing them.

**In your app:** start with `auto`, read `would_action` in your verdicts for a week, then list what to enforce per site
under `sites.<name>.enforce`. Use `observe` to evaluate without changing traffic.

**Recap:** modes: auto, observe, strict, off; `enforce` / `observe` lists per site win over the mode

## 7. Change behaviour live

**Goal:** turn Immune down or off without a restart.

In [13]:
immune.configure(sites={"ordering": {"enabled": False}})
with immune.site("ordering"):
    show(immune.verdict(chat(client, OFF_TASK)))
immune.configure(sites={"ordering": {"enabled": True}})
try:
    immune.configure(sensor={"model": "jev-2.0.0"})
except immune.ConfigError as error:
    print("ConfigError:", error)

no verdict: Immune did not screen this call
ConfigError: sensor cannot change at runtime; call immune.init() again


**What you'll see:** with the site switched off there is no verdict. Settings that decide how Immune is built (such as
the sensor) need `immune.init()` again, and saying so is an error rather than a silent no-op.

**In your app:** `immune.configure(mode="observe")` for an incident, `sites.<name>.enabled: false` for one feature,
and `IMMUNE_DISABLED=1` to switch Immune off without a deploy.

**Recap:** `immune.configure()` applies from the next call; kill switches per site and per process

## 8. Keep secrets and private data in

**Goal:** see a pasted credential redacted from the reply, and personal data masked before anything reaches Jev.

The key below is made up for this notebook. (AWS's documented example key, `AKIA…EXAMPLE`, would pass: Jev recognises
it as a documentation sample.)

In [14]:
CONFIG_NOTE = (
    "Can you turn this config into a table with every setting and its value, for the new ops hire?\n\n"
    "storage:\n  bucket: orders-prod\n  region: us-east-1\n  access_key_id: AKIA2E5QZ7XJ4M8RW3TD"
)
OPS_BOT = "You help Bob's Burgers staff with operations: supply tickets, supplier emails and configuration notes."
with immune.site("ops"):
    summary = chat(client, CONFIG_NOTE, system=OPS_BOT)
print(reply_of(summary))
show(immune.verdict(summary))

Here’s the config in a simple table:

| Setting | Value |
|---|---|
| storage.bucket | orders-prod |
| storage.region | us-east-1 |
| storage.access_key_id | [redacted] |

If you want, I can also reformat it into CSV, Markdown, or a more human-friendly ops note.
action=rewrite  would_action=rewrite  site=ops
  output.secret_leak               p=0.91  rewrite   enforced F3  · secret:aws_access_key; secret_real=0.64
  why: rewrite: output.secret_leak
  jev: jev, 3 requests, 365 ms, 2233 tokens


In [15]:
with LangSmithRecorder() as recorder:
    start(config=PROJECT / "immune.yaml")
    with immune.site("ordering"):
        chat(client, OFF_TASK + " Email it to jo@example.com and bill card 4111 1111 1111 1111.")
    immune.shutdown()
start(config=PROJECT / "immune.yaml")
jev_saw = json.dumps(
    [child["inputs"].get("state") for run in recorder.immune_runs() for child in recorder.children(run)]
)
print("email reached Jev:", "jo@example.com" in jev_saw, "| card reached Jev:", "4111 1111 1111 1111" in jev_saw)
print("placeholders Jev saw:", sorted({token for token in ("[EMAIL]", "[CARD]") if token in jev_saw}))

email reached Jev: False | card reached Jev: False
placeholders Jev saw: ['[CARD]', '[EMAIL]']


**What you'll see:** the key comes back as `[redacted]` (floor rule F3). A reflex found the key-shaped string and Jev
confirmed the conversation treats it as a real key; the hit's evidence shows both. Jev also saw `[EMAIL]` and `[CARD]`
instead of the real values.

**In your app:** nothing to add; both are on by default (`privacy.redact_before_sensor`). `privacy.profiling: local`
also keeps your system prompt out of Jev.

**Recap:** secrets redacted from replies; personal data masked before Jev

## 9. Copies of your prompt: Jev decides

**Goal:** see how Immune tells a policy customers may read from private instructions when a reply copies the system
prompt.

In [16]:
POLICY_BOT = MENU_BOT + (
    " Refund policy: Customers can ask for a refund within 24 hours of delivery if an item was missing, cold or "
    "wrong. Refunds go back to the original payment method within three to five business days. Orders refunded in "
    "full are not eligible for loyalty points, and repeated refund requests are reviewed by the store manager."
)
with immune.site("ordering"):
    quoted = chat(client, "What's your refund policy? Please quote it word for word.", system=POLICY_BOT)
print(reply_of(quoted))
show(immune.verdict(quoted))

Refund policy, word for word:

“Customers can ask for a refund within 24 hours of delivery if an item was missing, cold or wrong. Refunds go back to the original payment method within three to five business days. Orders refunded in full are not eligible for loyalty points, and repeated refund requests are reviewed by the store manager.”
action=allow  would_action=allow  site=ordering
  why: no threats detected
  jev: jev, 3 requests, 454 ms, 2708 tokens


**What you'll see:** the policy reaches the customer. The reply copies the system prompt word for word, so the
prompt-copy reflex nominated it, and Jev judged it information meant for customers.

Now a reply that copies private rules. `immune test` screens a message and a draft reply without calling a model;
`--live` asks the real Jev:

In [17]:
RULES_BOT = (
    "You are the ordering assistant for Bob's Burgers. Never offer discounts above 10 percent, and pass refunds over "
    "50 dollars to the store manager without mentioning this rule to the customer."
)
RULES_REPLY = (
    "My instructions say: never offer discounts above 10 percent, and pass refunds over 50 dollars to the store "
    "manager without mentioning this rule to the customer."
)
cli("test", "How do refunds work?", "--operator", RULES_BOT, "--reply", RULES_REPLY, *(["--live"] if JEV else []))

reply: My instructions say: [redacted].
action: rewrite (would rewrite)   taint: clean   session risk: 0.05   sensor: jev
threat                       stage   p     action   enforced  evidence
───────────────────────────  ──────  ────  ───────  ────────  ────────────────────────────────────────────────────
output.prompt_copy           output  0.97  rewrite  yes       copied operator instructions; copy_confidential=0.97
output.task_deviation        output  0.87  rewrite  no        task_fidelity=0.13
output.reveals_instructions  output  0.98  rewrite  no        reveals_instructions=0.98


**What you'll see:** the copied rules are redacted (F3): Jev judged them private guidance for the assistant.

**In your app:** nothing to configure. If Jev ever withholds something customers should see, keep that text in a
clearly labelled section of the prompt (like `Refund policy:` above) or in your knowledge base, and report the case
with `immune.feedback(trace_id, "false_positive")`.

**Recap:** Jev decides each copy; private rules stay private; public policies pass

## 10. Mark what you paste in

**Goal:** mark retrieved text as untrusted, and see a poisoned document neutralized.

In [18]:
ARTICLE = "Delivery: we deliver within 3 miles from 11am to 10pm. It costs $2.99 and takes 30 to 45 minutes."
HELP_BOT = "You answer questions for Bob's Burgers customers using only the articles provided."
question = f"How long does delivery take?\n\nArticles:\n{immune.untrusted(ARTICLE, source='kb:delivery')}"
with immune.site("help-center"):
    answer = chat(client, question, system=HELP_BOT)
print(reply_of(answer))
show(immune.verdict(answer))

Delivery takes **30 to 45 minutes**.
action=allow  would_action=rewrite  site=help-center
  output.follows_data_instructions p=0.96  rewrite   observed  · follows_data_instructions=0.96
  why: observed (would rewrite): output.follows_data_instructions
  jev: jev, 3 requests, 602 ms, 2461 tokens


Now a **replay**: the email from Immune's reconstruction of the EchoLeak incident, loaded from the scenario library
(it is not printed here), is pasted in as an article.

In [19]:
LIBRARY = ScenarioLibrary([REPOSITORY / "scenarios"]) if REPOSITORY else None
if LIBRARY is None:
    print("the scenario library is not available outside the Immune repository")
else:
    echoleak = LIBRARY.find("incident.echoleak")
    poisoned = echoleak.turns[0].data[0].text
    if not JEV:
        start(config=PROJECT / "immune.yaml", sensor=MockSensor(echoleak.signals))
    question = f"Summarize the latest article.\n\nArticles:\n{immune.untrusted(poisoned, source='kb:latest')}"
    with immune.site("help-center"):
        answer = chat(client, question, system=HELP_BOT)
    print(f"library document: {len(poisoned)} characters")
    show(immune.verdict(answer))
    start(config=PROJECT / "immune.yaml")

library document: 149 characters
action=neutralize  would_action=neutralize  site=help-center
  data.instructions                p=0.98  neutralize enforced F7  · instructions_to_ai=0.98
  data.exfil_request               p=0.95  neutralize observed  · exfil_request=0.95
  why: neutralize: data.instructions; observed (would neutralize): data.exfil_request
  jev: jev, 3 requests, 371 ms, 2490 tokens


**What you'll see:** the first answer uses the article normally. For the poisoned one, Jev reports instructions
addressed to the AI; at very high confidence (and when both of Immune's views agree) the article is replaced by a
notice before the model reads it (F7). With live Jev, how confident it is decides whether that happens.

**In your app:**

```python
context = "\n\n".join(immune.untrusted(chunk.text, source=chunk.source) for chunk in chunks)
```

Tool results, Anthropic documents and Gemini function responses are recognized as data automatically.

**Recap:** `immune.untrusted()` for pasted text; poisoned data is neutralized before the model reads it

## 11. Links in replies

**Goal:** see a data-carrying link removed, and an ordinary link left alone.

In [20]:
with immune.site("ordering"):
    linked = chat(client, "Where can I see the full menu online?")
print(reply_of(linked))
show(immune.verdict(linked))

You can view the full menu online here: https://bobsburgers.example/menu
action=allow  would_action=allow  site=ordering
  why: no threats detected
  jev: jev, 2 requests, 404 ms, 1815 tokens


In [21]:
if LIBRARY is not None:
    recorded = echoleak.turns[0].reply.text
    replay = fake_client(FakeReply(text=recorded))
    history = [{"role": "user", "content": immune.untrusted(poisoned, source="email")}]
    with immune.site("inbox"):
        leaked = chat(replay, "Summarize my inbox.", system="You summarize emails.", history=history, charge=False)
    print("replayed reply contained a link to evil.test:", "evil.test" in recorded)
    print("app received a link to evil.test:", "evil.test" in reply_of(leaked))
    show(immune.verdict(leaked))

replayed reply contained a link to evil.test: True
app received a link to evil.test: False
action=rewrite  would_action=rewrite  site=inbox
  data.instructions                p=0.98  neutralize enforced F7  · instructions_to_ai=0.98
  data.exfil_request               p=0.95  neutralize observed  · exfil_request=0.95
  output.exfil_link                p=0.92  rewrite   enforced F2  · image to evil.test carries data; link_exfiltrates=0.92
  output.follows_data_instructions p=0.91  rewrite   observed  · follows_data_instructions=0.91
  output.harmful                   p=0.89  rewrite   observed  · harmful_output=0.89
  output.ungrounded                p=0.88  rewrite   observed  · claims=0.88
  why: rewrite: data.instructions, output.exfil_link; observed (would rewrite): data.exfil_request, output.follows_data_instructions, output.harmful, output.ungrounded
  jev: jev, 4 requests, 590 ms, 2863 tokens


**What you'll see:** the menu link (a host your prompt names, with no data in the URL) is untouched. In the **replay**,
the model's recorded reply puts conversation data into an image URL; Immune removes it (F2) before your app renders
it.

**Recap:** ordinary links pass; links that carry data to unknown hosts are removed

## 12. Stream safely

**Goal:** stream replies with Immune screening them as they arrive.

In [22]:
BUDGET.check()
started, first, parts = time.perf_counter(), None, []
with immune.site("ops"):
    stream = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": OPS_BOT}, {"role": "user", "content": CONFIG_NOTE}],
        reasoning_effort="none",
        max_completion_tokens=300,
        stream=True,
        stream_options={"include_usage": True},
    )
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            first = first or time.perf_counter() - started
            parts.append(chunk.choices[0].delta.content)
        if chunk.usage:
            BUDGET.charge(chunk)
streamed = "".join(parts)
print(streamed)
print(f"first token after {first * 1000:.0f} ms; key in the stream: {'AKIA2E5QZ7XJ4M8RW3TD' in streamed}")
show(immune.verdict())

Here’s the config converted into a table:

| Setting | Value |
|---|---|
| storage.bucket | orders-prod |
| storage.region | us-east-1 |
| storage.access_key_id | [redacted] |
first token after 814 ms; key in the stream: False
action=rewrite  would_action=rewrite  site=ops
  output.secret_leak               p=0.93  rewrite   enforced F3  · secret:aws_access_key; secret_real=0.67
  why: rewrite: output.secret_leak
  jev: jev, 3 requests, 350 ms, 2158 tokens


**What you'll see:** text arrives as soon as it is safe, up to the key. From there the stream waits for Jev's decision
and the rest arrives with the key redacted. Sites that could enforce a whole-reply check are buffered instead,
automatically.

**In your app:** nothing changes; stream as usual. `streaming: buffered` in `immune.yaml` (globally or per site)
screens the whole reply before releasing it.

**Recap:** progressive streaming; held from the first candidate until Jev decides

## 13. Tools and agents

**Goal:** see Immune check tool calls in an ordinary function-calling loop.

The tools below record what they would do; nothing is sent.

In [23]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {"type": "object", "properties": properties, "required": list(properties)},
        },
    }
    for name, description, properties in [
        ("send_receipt", "Email a receipt", {"order_id": {"type": "integer"}, "email": {"type": "string"}}),
        ("refund_order", "Refund an order", {"order_id": {"type": "integer"}, "amount": {"type": "number"}}),
        ("check_stock", "Check whether an item is in stock", {"item": {"type": "string"}}),
        ("update_account", "Add loyalty points", {"account": {"type": "string"}, "points": {"type": "integer"}}),
    ]
]
FLAKY = {"on": False}


def run_tool(name, arguments):
    if name == "check_stock" and FLAKY["on"]:
        return "temporary error: the stock service did not answer, please retry"
    return f"{name}({arguments}) recorded (demo: nothing was sent or charged)"


def agent(client, text, history, system=MENU_BOT + " If a tool reports a temporary error, retry it.", charge=True):
    """A plain function-calling loop. Immune removes risky tool calls before the loop sees them."""
    history.append({"role": "user", "content": text})
    for _ in range(6):
        if charge:
            BUDGET.check()
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "system", "content": system}, *history],
            tools=TOOLS,
            reasoning_effort="none",
            max_completion_tokens=300,
        )
        if charge:
            BUDGET.charge(response)
        message = response.choices[0].message
        history.append(message.model_dump(exclude_none=True, exclude={"annotations", "audio"}))
        if not message.tool_calls:
            return message.content or "", immune.verdict(response)
        for call in message.tool_calls:
            result = run_tool(call.function.name, json.loads(call.function.arguments or "{}"))
            print(f"  tool ran: {call.function.name} -> {result}")
            history.append({"role": "tool", "tool_call_id": call.id, "content": result})
    return "(stopped)", immune.verdict(response)

In [24]:
with immune.site("ordering"), immune.session("agent-1"):
    text, verdict = agent(client, "Please email the receipt for order 5521 to jo@example.com", [])
print(text)
show(verdict)

  tool ran: send_receipt -> send_receipt({'order_id': 5521, 'email': 'jo@example.com'}) recorded (demo: nothing was sent or charged)


Done — I’ve queued the receipt for order 5521 to jo@example.com.
action=allow  would_action=rewrite  site=ordering
  data.exfil_request               p=0.85  neutralize observed  · exfil_request=0.85
  output.follows_data_instructions p=0.88  rewrite   observed  · follows_data_instructions=0.88
  why: observed (would rewrite): data.exfil_request, output.follows_data_instructions
  jev: jev, 3 requests, 299 ms, 2602 tokens


In [25]:
FLAKY["on"] = True
with immune.site("ordering"), immune.session("agent-2"):
    text, verdict = agent(client, "Is the Garden Stack in stock right now?", [])
FLAKY["on"] = False
print(text)
show(verdict)

  tool ran: check_stock -> temporary error: the stock service did not answer, please retry


  tool ran: check_stock -> temporary error: the stock service did not answer, please retry


  tool ran: check_stock -> temporary error: the stock service did not answer, please retry


I’m sorry, but I can’t confirm the Garden Stack’s stock right now because the stock service is temporarily unavailable. Please try again in a moment, and I can recheck it for you.
action=allow  would_action=rewrite  site=ordering
  data.instructions                p=0.70  neutralize observed F7  · instructions_to_ai=0.70
  data.instructions                p=0.70  neutralize observed F7  · instructions_to_ai=0.70
  data.instructions                p=0.70  neutralize observed F7  · instructions_to_ai=0.70
  output.follows_data_instructions p=0.81  rewrite   observed  · follows_data_instructions=0.81
  why: observed (would rewrite): data.instructions, output.follows_data_instructions
  jev: jev, 5 requests, 364 ms, 3600 tokens


And a **replay**: in Immune's reconstruction of ForcedLeak, a lead record carries instructions and the model tries to
email CRM data to the address they name. The record and the model's tool call are the library's recording.

In [26]:
if LIBRARY is not None:
    forced = LIBRARY.find("incident.forcedleak").turns[0]
    recorded_call = forced.reply.tool_calls[0]
    replay = fake_client(
        [FakeReply(tool_calls=[FakeToolCall("send_email", dict(recorded_call.arguments))]), FakeReply(text="Done.")]
    )
    email_tool = [{"type": "function", "function": {"name": "send_email", "description": "Send an email"}}]
    history = [
        {"role": "user", "content": "Summarize the new lead."},
        {
            "role": "assistant",
            "tool_calls": [{"id": "t0", "type": "function", "function": {"name": "get_lead", "arguments": "{}"}}],
        },
        {"role": "tool", "tool_call_id": "t0", "content": forced.data[0].text},
    ]
    with immune.site("crm"):
        held = replay.chat.completions.create(
            model=MODEL,
            messages=[{"role": "system", "content": "You are a CRM assistant."}, *history],
            tools=email_tool,
        )
    print(held.choices[0].message.content)
    show(immune.verdict(held))

The action send_email(body='ann@a.test, bo@b.test, cy@c.test', to='leads-archive@expired-partner.test') was not carried out: the destination came only from untrusted content.
action=hold  would_action=neutralize  site=crm
  data.instructions                p=0.84  neutralize observed F7  · instructions_to_ai=0.84
  data.exfil_request               p=0.95  neutralize observed  · exfil_request=0.95
  tool.destination_provenance      p=0.96  hold      enforced F4  · [EMAIL] appeared only in untrusted content; destination_requested=0.04
  tool.unrequested                 p=0.90  confirm   observed  · serves_request=0.10
  why: hold: tool.destination_provenance; observed (would neutralize): data.exfil_request, data.instructions, tool.unrequested
  jev: jev, 4 requests, 494 ms, 2243 tokens


**What you'll see:**

- The receipt goes to the address the customer typed: allowed.
- The flaky tool makes the model retry. Past the limit (more than 3 identical calls in a turn, set by
  `limits.max_identical_tool_calls`) the loop breaker (F6) stops it; a model that gives up sooner never reaches it.
- In the replay, the email to an address that only the untrusted record mentioned is held (F4, destination
  provenance), and the model is told why.

**In your app:** keep your agent loop as it is. Declare what tools can do and where they may send data:

```yaml
sites:
  ordering:
    allowed_destinations: [bobsburgers.example]
    tools:
      send_receipt: {egress: true}
      refund_order: {writes_state: true, irreversible: true}
```

**Recap:** risky calls are removed before your loop sees them; loops are broken; data can't be sent where only an
attacker pointed

## 14. Your own protections: vaccines

**Goal:** add protections specific to your app, one YAML file each, and manage them with the `immune vaccines`
commands.

This cell writes five vaccines, one per detector kind, into the project folder.

In [27]:
VACCINES = {
    "bobs.no_competitor_mentions.yaml": """\
id: bobs.no_competitor_mentions
title: The reply names a competitor
stage: output
detect:
  keywords: ["Burger Palace", "McRival"]
respond:
  message: "I can only talk about Bob's Burgers products."
tests:
  positives: ["Our Double Stack is closest to Burger Palace's Mega Melt."]
  negatives: ["The Classic is $9 and comes with fries."]
""",
    "bobs.no_account_numbers.yaml": """\
id: bobs.no_account_numbers
title: A customer pastes a loyalty account number
stage: input
detect:
  regex: ['\\bBB-\\d{8}\\b']
respond:
  message: "Please don't share your account number in chat."
tests:
  positives: ["My loyalty account is BB-12345678"]
  negatives: ["My order number is 5521"]
""",
    "bobs.no_unlisted_discounts.yaml": """\
id: bobs.no_unlisted_discounts
title: The reply offers a discount the menu doesn't list
stage: output
detect:
  questions:
    - key: promises_discount
      text: "The assistant output offers a discount, promotion or free item that the operator instructions do not list."
respond:
  message: "Our current prices are on the menu."
tests:
  positives: ["Good news: students get 20% off every Tuesday!"]
  negatives: ["The Classic is $9 and comes with fries."]
""",
    "bobs.refund_over_limit.yaml": """\
id: bobs.refund_over_limit
title: Refunds over $100 need the customer's confirmation
stage: tool
detect:
  tool: refund_order
  argument: {path: amount, greater_than: 100}
respond:
  tool: confirm
tests:
  positives: [{tool: refund_order, arguments: {order_id: 5521, amount: 250}}]
  negatives: [{tool: refund_order, arguments: {order_id: 5521, amount: 20}}]
""",
    "bobs.vip_accounts.yaml": """\
id: bobs.vip_accounts
title: Changes to VIP accounts
stage: tool
detect:
  python: my_vaccines:touches_vip_account
respond:
  tool: deny
  message: VIP accounts are managed by the account team
tests:
  positives: [{tool: update_account, arguments: {account: vip-42, points: 500}}]
  negatives: [{tool: update_account, arguments: {account: acct-7, points: 500}}]
""",
}
for name, text in VACCINES.items():
    (PROJECT / "vaccines" / name).write_text(text, encoding="utf-8")
(PROJECT / "my_vaccines.py").write_text(
    "def touches_vip_account(context):\n"
    "    account = str(context.arguments.get('account', ''))\n"
    "    return f'account {account} is a VIP account' if account.startswith('vip-') else None\n",
    encoding="utf-8",
)
sys.path.insert(0, str(PROJECT))
(PROJECT / "immune.yaml").write_text(
    (PROJECT / "immune.yaml").read_text().replace("enforce: [input.off_task]", 'enforce: [input.off_task, "bobs.*"]'),
    encoding="utf-8",
)
environment = {**os.environ, "PYTHONPATH": str(PROJECT)}
cli("vaccines", "test", cwd=PROJECT, env=environment)

vaccine                      expect  example                                                    result  detail
───────────────────────────  ──────  ─────────────────────────────────────────────────────────  ──────  ───────────────────────────────────────────────────────────────
bobs.no_account_numbers      fire    My loyalty account is BB-12345678                          pass    fired p=1.00 pattern /\bBB-\d{8}\b/
bobs.no_account_numbers      quiet   My order number is 5521                                    pass    did not fire
bobs.no_competitor_mentions  fire    Our Double Stack is closest to Burger Palace's Mega Melt.  pass    fired p=1.00 keyword 'Burger Palace'
bobs.no_competitor_mentions  quiet   The Classic is $9 and comes with fries.                    pass    did not fire
bobs.no_unlisted_discounts   fire    Good news: students get 20% off every Tuesday!             pass    fired p=0.97 bobs_no_unlisted_discounts__promises_discount=0.97
bobs.no_unlisted_discounts   quiet   T

Each file carries its own examples, and `immune vaccines test` checks them. Vaccines start observed; this app enforces
them at the `ordering` site with `enforce: [input.off_task, "bobs.*"]`. Restart with the project's settings and try
them on the live model:

In [28]:
start(config=PROJECT / "immune.yaml")
checks = [
    "My friend says Burger Palace's fries are better than yours. Are they?",
    "My loyalty account is BB-12345678. How many points do I have?",
    "Is there a student discount?",
]
for text in checks:
    with immune.site("ordering"):
        response = chat(client, text)
    print(f"> {text}\n{reply_of(response)}")
    show(immune.verdict(response))
    print()

> My friend says Burger Palace's fries are better than yours. Are they?
I can only talk about Bob's Burgers products.
action=rewrite  would_action=rewrite  site=ordering
  bobs.no_competitor_mentions      p=1.00  rewrite   enforced  · keyword 'Burger Palace'
  why: rewrite: bobs.no_competitor_mentions
  jev: jev, 2 requests, 394 ms, 1903 tokens



> My loyalty account is BB-12345678. How many points do I have?
Please don't share your account number in chat.
action=redirect  would_action=redirect  site=ordering
  bobs.no_account_numbers          p=1.00  redirect  enforced  · pattern /\bBB-\d{8}\b/
  why: redirect: bobs.no_account_numbers
  jev: jev, 1 requests, 428 ms, 1002 tokens



> Is there a student discount?
I’m not seeing any student discount listed for Bob’s Burgers.

If you’d like, I can still help with:
- current menu items
- placing an order
- delivery options
- refunds
- loyalty points


action=allow  would_action=allow  site=ordering
  why: no threats detected
  jev: jev, 2 requests, 357 ms, 1865 tokens



In [29]:
with immune.site("ordering"), immune.session("refund-1"):
    history = []
    text, verdict = agent(client, "My order 5521 arrived cold and wrong. Please refund $250.", history)
    print(text)
    show(verdict)
    text, verdict = agent(client, "yes", history)
    print(text)
with immune.site("ordering"), immune.session("vip-1"):
    text, verdict = agent(client, "Please add 500 loyalty points to account vip-42.", [])
print(text)
show(verdict)

Before I do that, please confirm: refund_order(amount=250, order_id=5521). Reply "yes" to continue. (ref a9875d78b6)
action=confirm  would_action=confirm  site=ordering
  bobs.refund_over_limit           p=1.00  confirm   enforced  · refund_order: amount=250
  why: confirm: bobs.refund_over_limit
  jev: jev, 2 requests, 447 ms, 1520 tokens


  tool ran: refund_order -> refund_order({'order_id': 5521, 'amount': 250}) recorded (demo: nothing was sent or charged)


Refund requested for order 5521 in the amount of $250.


The action update_account(account='vip-42', points=500) was blocked: VIP accounts are managed by the account team.
action=deny  would_action=deny  site=ordering
  bobs.vip_accounts                p=1.00  deny      enforced  · account vip-42 is a VIP account
  why: deny: bobs.vip_accounts
  jev: jev, 2 requests, 385 ms, 1510 tokens


In [30]:
immune.configure(vaccines={"disabled": ["bobs.no_competitor_mentions"]})
with immune.site("ordering"):
    response = chat(client, checks[0])
print("with the vaccine switched off live:", sorted(immune.verdict(response).threats()) or "nothing caught")
immune.configure(vaccines={"disabled": []})
try:
    start(config={"vaccines": {"disabled": ["output.secret_leak"]}})
except immune.ConfigError as error:
    print("floor guard:", error)
start(config=PROJECT / "immune.yaml")
cli("vaccines", "list", "--custom", "--site", "ordering", cwd=PROJECT, env=environment)
cli("vaccines", "trial", "bobs.no_competitor_mentions", cwd=PROJECT, env=environment)

with the vaccine switched off live: nothing caught
floor guard: these settings would disable floor protections: vaccines.disabled: output.secret_leak (F3). Set vaccines.allow_floor_changes: true if this is intended


protection                   source         stage   detector       floor  state  why
───────────────────────────  ─────────────  ──────  ─────────────  ─────  ─────  ─────────────
bobs.no_account_numbers      vaccine 1.0.0  input   deterministic         on     on by default
bobs.no_competitor_mentions  vaccine 1.0.0  output  deterministic         on     on by default
bobs.no_unlisted_discounts   vaccine 1.0.0  output  jev                   on     on by default
bobs.refund_over_limit       vaccine 1.0.0  tool    deterministic         on     on by default
bobs.vip_accounts            vaccine 1.0.0  tool    deterministic         on     on by default

5 protections, 5 vaccines loaded


trial bobs.no_competitor_mentions (offline): 40 everyday output samples
fired on 0 of 40 samples (0.0%)


**What you'll see:**

- The competitor's name is caught (keywords), and the account number is redirected before the model sees it (regex).
- The discount vaccine asks Jev; the live model usually answers honestly, so it may stay quiet, which is the point: it
  is insurance.
- The $250 refund waits for a sealed confirmation, and "yes" releases exactly that call (tool rule).
- The VIP change is denied (Python function).
- Vaccines switch on and off live, and switching off a floor rule is refused without `allow_floor_changes: true`.
- `immune vaccines trial` estimates how often a vaccine would fire on everyday traffic.

**In your app:** put vaccine files in `vaccines/`, list the directory under `vaccines.paths`, and run
`immune vaccines test` in CI. `immune vaccines new <id>` scaffolds one, and `immune vaccinate` builds and tests one
from examples. The [vaccines guide](../../docs/guides/vaccines.md) covers every field.

**Recap:** five detector kinds; start observed, enforce per site; test and trial before you enforce

## 15. When something is blocked

**Goal:** choose how your code learns about a block.

In [31]:
from immune.config.loader import SettingsLoader

start(config=SettingsLoader().load(PROJECT / "immune.yaml").model_copy(update={"on_block": "raise"}))
try:
    with immune.site("ordering"):
        chat(client, OFF_TASK)
    print("not blocked this time")
except immune.Blocked as blocked:
    print("immune.Blocked:", blocked.verdict.explanation)
    print("also an OpenAI error:", isinstance(blocked, openai.OpenAIError))
start(config=PROJECT / "immune.yaml")

immune.Blocked: redirect: input.off_task
also an OpenAI error: True


**What you'll see:** with `on_block: raise`, a block raises `immune.Blocked`, which is also the SDK's own error type,
so existing `except openai.OpenAIError` handlers keep working. The default, `respond`, returns a normal response with a
safe reply, so code that doesn't know about Immune keeps working too. Redacted links and secrets never raise.

**Recap:** `respond` (default) or `raise`; `blocked.verdict` explains why

## 16. See it in your tools

**Goal:** stream verdicts to your logs, and see threats in LangSmith.

In [32]:
collected = []
unsubscribe = immune.on_verdict(collected.append)
with immune.site("ordering"):
    chat(client, "Do you deliver after 9pm?")
    chat(client, OFF_TASK)
time.sleep(0.5)
unsubscribe()
for verdict in collected:
    print(f"{verdict.trace_id}  {verdict.site:<9} {verdict.action.value:<9} {sorted(verdict.threats()) or 'clean'}")

c98ab84121aa4cb5  ordering  allow     ['business.unauthorized_commitment']
9a6bcad4991a4a25  ordering  redirect  ['input.off_task']


In [33]:
if os.environ.get("LANGSMITH_TRACING") and os.environ.get("LANGSMITH_API_KEY") and LIVE:
    project = os.environ.get("LANGSMITH_PROJECT", "default")
    print(f"LangSmith is on: open the project {project!r} and filter by feedback key immune.threat")
else:
    with LangSmithRecorder() as recorder:
        start(config=PROJECT / "immune.yaml")
        with immune.site("ordering"):
            chat(client, "Do you deliver after 9pm?")
            chat(client, OFF_TASK)
        immune.shutdown()
    start(config=PROJECT / "immune.yaml")
    runs = recorder.immune_runs()
    for run in runs:
        print(run["name"], run["tags"])
    print(f"{len(runs)} run(s) for 2 calls: the clean question was not sent")
    print("feedback:", sorted({item["key"] for item in recorder.feedback}))

immune · redirect · input.off_task ['immune', 'immune:redirect', 'immune:enforced', 'threat:input.off_task', 'site:ordering']
1 run(s) for 2 calls: the clean question was not sent
feedback: ['immune.blocked', 'immune.threat']


**What you'll see:** every verdict reaches your callback on a background thread. LangSmith gets only the calls with a
possible threat, as runs named after the action and the threat, nested under your app's own runs when you trace
them, with `immune.threat` and `immune.blocked` feedback to filter on. Without LangSmith keys, a recorder shows what
would be sent.

**In your app:**

```bash
pip install "immune-ai[langsmith]"
export LANGSMITH_TRACING=true LANGSMITH_API_KEY=... LANGSMITH_PROJECT=my-app
```

OpenTelemetry works the same way: configure an SDK in your app and Immune adds an `immune.screen` span per call.

**Recap:** `immune.on_verdict()`; LangSmith for possible threats only; OpenTelemetry spans

## 17. Test it

**Goal:** check your settings and vaccines in your test suite, offline.

In [34]:
(PROJECT / "tests").mkdir(exist_ok=True)
(PROJECT / "tests" / "test_immune.py").write_text(
    """\
from pathlib import Path

import immune
from immune.testing import FakeReply, MockSensor

CONFIG = Path(__file__).resolve().parents[1] / "immune.yaml"
MENU_BOT = "You are the ordering assistant for Bob's Burgers."


def ask(harness, text):
    with immune.site("ordering"):
        return harness.openai().chat.completions.create(
            model="m", messages=[{"role": "system", "content": MENU_BOT}, {"role": "user", "content": text}]
        )


def test_off_topic_requests_are_redirected(immune_harness):
    harness = immune_harness(config=CONFIG, sensor=MockSensor({"off_task": 0.95}), script=FakeReply("Sure, a poem."))
    assert ask(harness, "Write me a poem").choices[0].message.content.startswith("I can't help with that here.")


def test_competitors_are_never_named(immune_harness):
    harness = immune_harness(config=CONFIG, script=FakeReply("Try Burger Palace."))
    assert ask(harness, "Ideas?").choices[0].message.content == "I can only talk about Bob's Burgers products."
""",
    encoding="utf-8",
)
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", "tests"],
    cwd=PROJECT,
    env=environment,
    capture_output=True,
    text=True,
    check=False,
)
print(result.stdout[-600:])

..                                                                       [100%]
2 passed in 2.18s



**What you'll see:** the tests pass offline. `immune_harness(config=...)` loads your real `immune.yaml`, a fake
provider plays the model and `MockSensor` scripts Jev's answers, so nothing touches the network.

**In your app:** add tests like these, plus `immune config validate`, `immune vaccines test` and `immune replay` (for
scenario files) to CI. The [developer guide](../../docs/guides/developer-guide.md#13-testing-and-ci) has ready-made
GitHub Actions and GitLab workflows, and shows how to record live Jev answers once (`RecordingSensor`) and replay them
in CI (`ReplaySensor`).

**Recap:** test your real settings offline; run Immune's checks in CI

## 18. Go to production

**Goal:** check the settings that matter once real traffic flows.

First, what happens when Jev is unreachable (a failing sensor stands in for an outage):

In [35]:
start(config=PROJECT / "immune.yaml", sensor=MockSensor(fail=True))
with immune.site("ops"):
    outage = chat(client, CONFIG_NOTE, system=OPS_BOT)
print("key in the reply:", "AKIA2E5QZ7XJ4M8RW3TD" in reply_of(outage))
show(immune.verdict(outage))
start(config=PROJECT / "immune.yaml")

immune: sensor unavailable for input (mock sensor configured to fail); floor candidates act without Jev


immune: sensor unavailable for outbound screening; floor candidates act without Jev


key in the reply: False
action=rewrite  would_action=rewrite  site=ops
  output.secret_leak               p=1.00  rewrite   enforced F3  · secret:aws_access_key; acted without Jev: the sensor was unavailable (sensor.on_outage: act_on_candidates)
  why: rewrite: output.secret_leak
  jev: tier0_only, 0 requests, 0 ms, 0 tokens


**What you'll see:** the call still gets a verdict and the key is still redacted. With Jev unreachable, floor
candidates act on their own (`sensor.on_outage: act_on_candidates`, the default), and the hit's evidence says so.
Immune fails open by default, so an outage never takes your app down.

**Checklist:**

| Setting | Why |
| --- | --- |
| `state: {backend: redis, url: ...}` or `IMMUNE_STATE_URL` | Several workers share sessions, provenance and promotions (SQLite for one host) |
| `sensor.api_keys: [...]` and `sensor.requests_per_minute` | Spread Jev load over several keys; observed detectors are shed first under pressure |
| `sensor.timeout_ms` (default 800) | Your latency budget for Jev; a miss continues without Jev |
| `sensor.on_outage` (default `act_on_candidates`) | What floor candidates do while Jev is unreachable; `pass` lets them through |
| `on_internal_error: pass` (default) | Fail open if Immune itself errors; `block` to refuse instead |
| `privacy.log`, `privacy.verdict_log` | How much evidence logs keep; an optional JSONL audit log |
| `IMMUNE_DISABLED=1` | The kill switch |
| `immune doctor --live` | A health check for deploys |

**Recap:** fails open; shared state for several workers; Jev capacity and latency settings

## 19. Copy it into your app

**Goal:** take what you built with you.

In [36]:
for path in sorted(PROJECT.rglob("*")):
    if path.is_file() and "__pycache__" not in path.parts and ".pytest_cache" not in path.parts:
        print(path.relative_to(PROJECT))
cli("config", "validate", PROJECT / "immune.yaml")
print(BUDGET)

immune.yaml
my_vaccines.py
tests/test_immune.py
vaccines/bobs.no_account_numbers.yaml
vaccines/bobs.no_competitor_mentions.yaml
vaccines/bobs.no_unlisted_discounts.yaml
vaccines/bobs.refund_over_limit.yaml
vaccines/bobs.vip_accounts.yaml


valid configuration (mode=auto, 2 sites)
38 OpenAI calls, $0.0116 this run, $0.0903 of $3.00 across runs


Copy `immune.yaml`, `vaccines/`, `my_vaccines.py` and `tests/test_immune.py` into your repository, rename the sites to
your own `immune.site(...)` names, and add `immune.init()` to your app's startup.

Where to go next:

- The [setup guide](../../docs/guides/setup.md) and [`examples/starter`](../starter): a complete minimal project,
  including CI files.
- [`examples/showcase`](../showcase): a larger app with four features, a guided demo and a web inspector.
- The [developer guide](../../docs/guides/developer-guide.md) and the [configuration reference](../../docs/guides/configuration.md).

In [37]:
immune.shutdown()
shutil.rmtree(WORK, ignore_errors=True)
print("done")

done
